In [ ]:
import cv2
import numpy as np
import argparse
import yaml
import time
import copy

import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
import torchvision
import matplotlib.pyplot as plt

# parser = argparse.ArgumentParser()
# # parser.add_argument('--config', default='config.yaml')
# parser.add_argument('--config',default='./content/config_mymodel.yaml')




In [ ]:
class Average(object):
    """
    modified and influenced from DL class
    calculates average and current values
    """
    def __init__(self):
        self.reset()

    def reset(self):
      '''set up vars for data collection'''
        self.count = 0
        self.val = 0
        self.sum = 0
        self.avg = 0

    def update(self, val, n=1):
        '''update vars per batch'''
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
def accuracy(output, target):
    """
    modified and influenced from DL class
    calculates accuracy
    """
    #get batch
    batch_size = target.shape[0]
    #get prediction
    max, prediction = torch.max(output, dim=-1)
    #num correct
    correct = prediction.eq(target).sum() * 1.0
    #accuracy
    accuracy = correct / batch_size

    return accuracy

In [ ]:
def train(epoch, data_loader, model, optimizer, criterion):

    losses = Average()
    acc = Average()

    for idx, (data, target) in enumerate(data_loader):

        if torch.cuda.is_available():
            data = data.cuda()
            target = target.cuda()

        #https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#sphx-glr-beginner-blitz-autograd-tutorial-py
        #https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html\

        # zero the parameter gradients
        optimizer.zero_grad()

        #forward
        out = model(data)
        # print(f'output: ',out.shape)

        #loss
        loss = criterion(out, target)
        # print(f'loss: ',loss.shape)

        #backward
        loss.backward()

        #optimizer
        optimizer.step()

        #modified and reused form DL class
        batch_acc = accuracy(out, target)
        losses.update(loss, out.shape[0])
        acc.update(batch_acc, out.shape[0])

        #calculate stats
        if idx % 10 == 0:
            print(('Epoch: [{0}][{1}/{2}]\t'
                   'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                   'Prec @1 {top1.val:.4f} ({top1.avg:.4f})\t')
                   .format(epoch, idx, len(data_loader), loss=losses, top1=acc))



In [ ]:
def validate(epoch, val_loader, model, criterion):
    losses = Average()
    acc = Average()

    num_class = 11
    confusion_matrix =torch.zeros(num_class, num_class)
    # evaluation loop
    for idx, (data, target) in enumerate(val_loader):
        start = time.time()

        if torch.cuda.is_available():
            data = data.cuda()
            target = target.cuda()

        #https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#sphx-glr-beginner-blitz-autograd-tutorial-py
        #https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html\


        #forward
        out = model(data)
        #print(f'target: ',target.shape)
        #print(f'output: ',out.shape)
        #loss
        with torch.no_grad():
            loss = criterion(out, target)
        #print(f'loss: ',loss.shape)

        batch_acc = accuracy(out, target)

        # update confusion matrix
        _, preds = torch.max(out, 1)
        for t, p in zip(target.view(-1), preds.view(-1)):
            confusion_matrix[t.long(), p.long()] += 1

        losses.update(loss, out.shape[0])
        acc.update(batch_acc, out.shape[0])

        #modified and reused from DL class
        #calculate stats
        if idx % 10 == 0:
            print(('Epoch: [{0}][{1}/{2}]\t'
               'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
               'Prec @1 {top.val:.4f} ({top.avg:.4f})\t')
               .format(epoch, idx, len(val_loader), loss=losses, top=acc))

    #calculate stats
    confusion_matrix = confusion_matrix / confusion_matrix.sum(1)
    per_cls_acc = confusion_matrix.diag().detach().numpy().tolist()
    for i, acc_i in enumerate(per_cls_acc):
        print("Accuracy of Class {}: {:.4f}".format(i, acc_i))

    print("Accuracy: {top.avg:.4f}".format(top=acc))
    return acc.avg, confusion_matrix

In [ ]:
def adjust_learning_rate(optimizer, epoch, args):
    #modified and influenced from DL class
    epoch += 1
    if epoch <= args['warmup']:
        lr = args['learning_rate'] * epoch / args['warmup']
    elif epoch > args['steps'][1]:
        lr = args['learning_rate'] * 0.01
    elif epoch > args['steps'][0]:
        lr = args['learning_rate'] * 0.1
    else:
        lr = args['learning_rate']
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

        #self.conv_layer = nn.Sequential(
        self.conv1= nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.batchnorm1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batchnorm2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(128)
        self.conv4= nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.batchnorm4 = nn.BatchNorm2d(128)
        self.conv5= nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.batchnorm5 = nn.BatchNorm2d(256)
        self.conv6= nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)

        self.avgpool = nn.AvgPool2d(kernel_size=1, stride=1)
        self.drop1 = nn.Dropout2d(p=0.01)
        self.linear1 = nn.Linear(256 *4 *4, 1024)
        self.linear2 = nn.Linear(1024, 512)
        self.drop2 = nn.Dropout(p=0.01)
        self.linear3 = nn.Linear(512, 11)
        #)


    def forward(self, x):
        outs = None

        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        # x = self.pool(x)

        x = self.conv2(x)
        x = self.batchnorm2(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = self.drop1(x)

        x = self.conv3(x)
        x = self.batchnorm3(x)
        x = self.relu(x)
        # x = self.pool(x)

        x = self.conv4(x)
        x = self.batchnorm4(x)
        x = self.relu(x)
        x = self.pool(x)
        #x = self.drop1(x)

        x = self.conv5(x)
        x = self.batchnorm5(x)
        x = self.relu(x)

        x = self.conv6(x)
        x = self.relu(x)
        x = self.pool(x)

        #print(x.shape)
        # x = x.view(-1, 256 * 2 * 2)
        x = torch.flatten(x, 1)
        x = self.drop2(x)
        # print(x.shape)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.relu(x)
        #x = self.drop2(x)
        x = self.linear3(x)
        outs = x

        return outs

In [ ]:
def setup_data():

  # Define the data transforms
  transform = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
  ])

  # Load the SVHN dataset
  svhn_train = datasets.SVHN('./svhn', split='train', download=True, transform=transform)

  svhn_train, svhn_val = torch.utils.data.random_split(svhn_train, [0.8,0.2])

  svhn_test = datasets.SVHN('./svhn', split='test', download=True, transform=transform)

  cifar = datasets.CIFAR10('./cifar10', train=True, download=True, transform=transform)
  cifar.targets[:] = [10]*len(cifar)

  cifar_train, cifar_val, cifar_test, _ = torch.utils.data.random_split(cifar, [0.2,0.05,.05,.7])

  # print(len(cifar.targets))
  train_set = torch.utils.data.ConcatDataset([svhn_train, cifar_train])
  val_set = torch.utils.data.ConcatDataset([svhn_val, cifar_val])
  test_set = torch.utils.data.ConcatDataset([svhn_test, cifar_test])
  # print(len(comb_datasets))

  # Create a PyTorch DataLoader for the dataset
  dataloader_train = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

  dataloader_val = torch.utils.data.DataLoader(val_set, batch_size=64, shuffle=True)

  dataloader_test = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=True)



  return dataloader_train, dataloader_val, dataloader_test

In [ ]:
def main():
    global args
    # args = parser.parse_args()
    # with open(args.config) as f:
    #     config = yaml.safe_load(f)

    # for key in config:
    #     for k, v in config[key].items():
    #         setattr(args, k, v)

    args = {'batch_size': 128, #128
    'learning_rate': 0.001, #0.0001
    'reg': 0.0005, #0.0005
    'epochs': 10, #10
    'steps': [6, 8], #[6, 8]
    'warmup': 0, #0
    'momentum': 0.9, #0.9
    # 'model': 'MyModel',
    'model': 'vgg16',
    'train': True,
    'loss_type': 'CE'}



    dataloader_train, dataloader_val, dataloader_test = setup_data()


    if args['model'] == 'vgg16':
        model = torchvision.models.vgg16(weights='VGG16_Weights.DEFAULT')

        model.classifier = nn.Sequential(*list(model.classifier.children())[:-1])
        num_classes = 11
        model.classifier[-1] = nn.Linear(4096, num_classes)

    elif args['model'] == 'MyModel':
        model = MyModel()




    if torch.cuda.is_available():
        model = model.cuda()


    criterion = nn.CrossEntropyLoss()


    optimizer = torch.optim.SGD(model.parameters(), args['learning_rate'],
                                momentum=args['momentum'],
                                weight_decay=args['reg'])
    best = 0.0
    best_cm = None
    best_model = None
    for epoch in range(args['epochs']):
        adjust_learning_rate(optimizer, epoch, args)

        # train loop
        train(epoch, dataloader_train, model, optimizer, criterion)

        # validation loop
        acc, cm = validate(epoch, dataloader_val, model, criterion)

        #test loop
        tacc, tcm = validate(epoch, dataloader_test, model, criterion)

        if acc > best:
            Tbest = tacc
            best = acc
            best_cm = cm
            best_model = copy.deepcopy(model)

    print('Best Prec @1 Acccuracy Valadation: {:.4f}'.format(best))
    print('Best Prec @1 Acccuracy Test: {:.4f}'.format(Tbest))
    per_cls_acc = best_cm.diag().detach().numpy().tolist()
    for i, acc_i in enumerate(per_cls_acc):
        print("Accuracy of Class {}: {:.4f}".format(i, acc_i))

    if args['train']:
        torch.save(best_model.state_dict(), './' + args['model'].lower() + '.pth')

# !python main.py --config configs/mymodel.yaml
# if __name__ == '__main__':
main()

NameError: name 'setup_data' is not defined

In [ ]:
# Iterate over the batches of data
for images, labels in dataloader:


    images = images.permute(0, 2, 3, 1).numpy()
    # print(images.shape)

    for image in images:
      # img = images[i, :, :, :]

      # print(image.shape)
      img = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
      # print(img.dtype)
      # img = img.astype(np.uint8)
      img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8UC3)#CV_8U
      # plt.imshow(img, cmap='gray')
      # plt.show()



      img_blur = cv2.GaussianBlur(img, (5,5),0) # blur


      # delta = 4
      # min_area=400
      # max_area=10000

      # mser = cv2.MSER_create(delta=delta, min_area = min_area, max_area = max_area)
      # regions = mser.detectRegions(img)
      # # regions = mser.detectRegions(img_gray)

      # color = (255, 0, 0)
      # thickness = 2
      # for region in regions[1]:
      #   # print(region)
      #   start_point = (region[0], region[1])
      #   end_point = (region[0] + region[2], region[1] + region[3])
      #   cv2.rectangle(img, start_point, end_point, color, thickness)


      # Set up the MSER parameters
      mser = cv2.MSER_create()

      # Detect MSER regions in the image
      regions, boundingbox = mser.detectRegions(img)

      # Filter out small regions
      min_area = 800
      # regions = [r for r in boundingbox if cv2.contourArea(r) > min_area]
      for region in regions:
          x, y, w, h = cv2.boundingRect(region)
          img =cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)

      for box in boundingbox:
        x, y, w, h = box;
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 1)
      # Draw the regions on the image


      # Show the image with the detected regions
      plt.imshow(img)
      plt.show()